# Train Arm C (blt_entropy_patching_syllable_seeded) at 5000 steps / 0.3GB (scale-up, question 1.3)

See `plans/PLAN.md` question 1.3, `phases/phase-2-small-train.md`. Uses the new
checkpoint/resume + LR schedule + grad clip + eval infra in `vislm/train.py`.
CODE_DIR is detected at runtime (Kaggle's dataset mount path is inconsistent between
runs - see `memory/project_kaggle_environment_quirks`).

**Rerun** with the corrected `entropy_threshold` (3.7 -> 3.0, see phase doc "Phát hiện:
entropy_threshold bị hiệu chỉnh sai") - already baked into `1_3_arm_C_blt_syllable.yaml`,
no override change needed here.


In [ ]:
import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_name or "(none detected)")

if "P100" in gpu_name:
    print("P100 detected - pinning torch==2.7.1+cu126 (last version supporting sm_60)")
    subprocess.run(
        ["pip", "install", "-q", "torch==2.7.1", "--index-url",
         "https://download.pytorch.org/whl/cu126"],
        check=True,
    )
else:
    print("Not a P100 - keeping the pre-installed torch build")

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
import os

_candidates = [
    "/kaggle/input/vislm-research-code",
    "/kaggle/input/datasets/nguyennn263/vislm-research-code",
]
CODE_DIR = next(p for p in _candidates if os.path.isdir(p))
os.environ["PYTHONPATH"] = CODE_DIR
print("CODE_DIR:", CODE_DIR)


In [ ]:
!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python {CODE_DIR}/setup/download_prepare_data.py \
  --target-gb 0.3 --out-dir /kaggle/working/data/prepared/fineweb2_vi --shard-size-mb 50


In [ ]:
!python -m vislm.train {CODE_DIR}/pillar1_configs/1_3_arm_C_blt_syllable.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_C_5000 \
  train.max_steps=5000 train.warmup_steps=200 \
  train.save_every=1000 train.eval_every=500 train.entropy_pretrain_steps=500


In [ ]:
import json

train_losses, val_losses = [], []
with open("/kaggle/working/runs/arm_C_5000/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            train_losses.append((row["step"], row["loss"]))
        if "eval_step" in row:
            val_losses.append((row["eval_step"], row["val_loss"]))

summary = {
    "n_steps": len(train_losses),
    "first_loss": train_losses[0][1],
    "last_loss": train_losses[-1][1],
    "min_loss": min(l for _, l in train_losses),
    "last_20_avg": sum(l for _, l in train_losses[-20:]) / len(train_losses[-20:]),
    "val_loss_curve": val_losses,
}
print(summary)

with open("/kaggle/working/metrics_arm_C_5000.jsonl", "w") as f:
    f.write(json.dumps({"section": "arm_C_5000", "results": summary}) + "\n")
